# Wind fleet: generation model, day-ahead forecast error and imbalance exposure (fixed)

Corrected version. Each *Fix* note says what changed and why.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

pd.set_option("display.width", 120)
rng = np.random.default_rng(42)

df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"])
P_RATED = 10_000
CUT_IN, V_RATED, CUT_OUT = 3.5, 12.0, 25.0

*Fix 0 — what height is `wind_ms` measured at?* The data dictionary does not say. The mock
asserted "10 m met mast" and scaled by (100/10)^0.14 = 1.38, which nearly doubles the energy
(cubic curve). Until the height is confirmed, treat the column as hub-height wind and carry
the shear case as an upside scenario. This is the biggest single uncertainty in the quote and
should be the first question back to the data owner.

In [2]:
SHEAR = 0.14
df["v_hub"] = df["wind_ms"]                                   # base case: as measured
df["v_hub_shear"] = df["wind_ms"] * (100 / 10) ** SHEAR       # upside case
df[["v_hub", "v_hub_shear"]].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
v_hub,17520.0,7.30,2.50,0.0,5.56,7.22,8.95,15.99
v_hub_shear,17520.0,10.07,3.45,0.0,7.67,9.97,12.36,22.07


*Fix 1 — `np.interp` needs sorted `xp`.* The manufacturer points were in insertion order,
so the interpolation was garbage (zero up to 14 m/s, then one). Sort the table and check
monotonicity before using it.

In [3]:
curve_pts = {0.0: 0.0, 3.5: 0.0, 25.0: 0.0, 12.0: 1.0, 5.0: 0.07, 7.0: 0.20, 9.0: 0.42, 10.5: 0.67, 11.5: 0.92, 14.0: 1.0}
tab = pd.Series(curve_pts).sort_index()
assert tab.index.is_monotonic_increasing
xp, fp = tab.index.to_numpy(), tab.to_numpy()
gen_tab = np.interp(df["v_hub"], xp, fp) * P_RATED
print("capacity factor from (sorted) tabulated curve:", round(gen_tab.mean() / P_RATED, 3))
tab

capacity factor from (sorted) tabulated curve: 0.299


0.0     0.00
3.5     0.00
5.0     0.07
7.0     0.20
9.0     0.42
10.5    0.67
11.5    0.92
12.0    1.00
14.0    1.00
25.0    0.00
dtype: float64

*Fix 2 — the analytic curve was rated at 9 m/s, the manufacturer table at 12.* A rated
speed of 9 m/s puts 60% of hours at full output and gives an 80% capacity factor, which no
wind portfolio achieves (UK offshore is ~40%, onshore ~27%). Use the table's rated speed,
keep the cap, and add cut-out (never triggers here: max hub speed 22 m/s, which is worth
saying rather than assuming).

*Fix 3 — a 10 GW fleet is not one turbine.* Sites see different speeds at the same hour, so
the fleet curve is the turbine curve smoothed over a spread of speeds (here sd 1.5 m/s).
That lowers the slope, which is what turns wind-speed error into MW error.

In [4]:
def turbine_curve(v, v_rated=V_RATED):
    v = np.asarray(v, dtype=float)
    p = np.clip((v / v_rated) ** 3, 0, 1)
    p = np.where(v < CUT_IN, 0.0, p)
    return np.where(v >= CUT_OUT, 0.0, p)

def fleet_curve(v, spread=1.5, n=41):
    v = np.asarray(v, dtype=float)[:, None]
    offs = np.linspace(-3, 3, n) * spread
    w = np.exp(-0.5 * (offs / spread) ** 2); w /= w.sum()
    return (turbine_curve(v + offs) * w).sum(axis=1)

df["gen_mw"] = fleet_curve(df["v_hub"]) * P_RATED
cf = df["gen_mw"].mean() / P_RATED
print(f"capacity factor, turbine curve: {turbine_curve(df['v_hub']).mean():.3f}   fleet curve: {cf:.3f}")
print(f"capacity factor if the data were 10 m wind (shear case): {fleet_curve(df['v_hub_shear']).mean():.3f}")
print("max wind:", round(df["v_hub"].max(), 1), "m/s; hours above cut-out:", int((df["v_hub"] >= CUT_OUT).sum()))

capacity factor, turbine curve: 0.294   fleet curve: 0.310
capacity factor if the data were 10 m wind (shear case): 0.581
max wind: 16.0 m/s; hours above cut-out: 0


*Fix 4 — "actual plus 0.5 m/s iid noise" is not a day-ahead forecast.* A real forecast has an
issue time, its error grows with horizon, is autocorrelated across hours (a mis-timed front
is wrong for many hours in a row) and is biased. Emulate a forecast issued at 12:00 on D-1
for every hour of D (horizons 12–35 h) with AR(1) error whose sd grows with horizon.
The absolute level is a guess; the *structure* is what matters for the exposure.

In [5]:
h = df["time"].dt.hour.to_numpy() + 12                         # horizon from the 12:00 D-1 origin
sd_h = 1.0 + 0.05 * h                                          # 1.6 m/s at 12 h, 2.75 m/s at 35 h
e = np.zeros(len(df))
z = rng.normal(0, 1, len(df))
for t in range(1, len(df)):
    e[t] = 0.85 * e[t - 1] + np.sqrt(1 - 0.85 ** 2) * z[t]
df["v_fc"] = np.clip(df["v_hub"] + e * sd_h + 0.3, 0, None)
df["gen_fc_mw"] = fleet_curve(df["v_fc"]) * P_RATED
print("wind-speed forecast error sd:", round((df["v_fc"] - df["v_hub"]).std(), 2), "m/s; bias:", round((df["v_fc"] - df["v_hub"]).mean(), 2))

wind-speed forecast error sd: 2.2 m/s; bias: 0.28


*Fix 5 — merge in UTC.* The mock converted actuals to Europe/London wall-clock and the
forecast to UTC wall-clock, then merged on the naive timestamps: every BST hour was paired
with the forecast for the following hour, which is why summer RMSE was 2–3x winter. Keep
both sides tz-aware in UTC (or convert both the same way) and check the join preserves the
row count *and* that a known hour lines up.

In [6]:
actual = df[["time", "gen_mw"]]
forecast = df[["time", "v_fc", "gen_fc_mw"]]
m = actual.merge(forecast, on="time", how="inner", validate="one_to_one")
assert len(m) == len(df)
m["month"] = m["time"].dt.month
m["year"] = m["time"].dt.year

*Fix 6 — calibrate on the past, evaluate on the future.* Fit the bias correction on 2022 and
report errors on 2023 only. In-sample RMSE flatters the forecast.

*Fix 7 — always compute a persistence baseline.* At 12:00 on D-1 the latest observation of
hour *h* is on D-1 for h ≤ 12 and on D-2 otherwise.

In [7]:
train, test = m[m["year"] == 2022], m[m["year"] == 2023].copy()
lr = LinearRegression().fit(train[["gen_fc_mw"]], train["gen_mw"])
test["gen_cal_mw"] = lr.predict(test[["gen_fc_mw"]])
print("calibration slope, intercept (fit on 2022):", round(lr.coef_[0], 3), round(lr.intercept_, 1))

gen = m.set_index("time")["gen_mw"]
hour = gen.index.hour
persist = pd.Series(np.where(hour <= 12, gen.shift(24), gen.shift(48)), index=gen.index)
test["persist_mw"] = persist.reindex(test["time"]).to_numpy()

def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
mean_out = test["gen_mw"].mean()
rows = {}
for name in ["gen_fc_mw", "gen_cal_mw", "persist_mw"]:
    r = rmse(test["gen_mw"], test[name])
    rows[name] = {"RMSE MW": round(r), "% of capacity": round(r / P_RATED, 3), "% of mean output": round(r / mean_out, 3),
                  "bias MW": round((test[name] - test["gen_mw"]).mean())}
res = pd.DataFrame(rows).T
res

calibration slope, intercept (fit on 2022): 0.601 1063.9


,RMSE MW,% of capacity,% of mean output,bias MW
gen_fc_mw,2113.0,0.211,0.736,485.0
gen_cal_mw,1642.0,0.164,0.572,208.0
persist_mw,2868.0,0.287,0.999,13.0


*Fix 8 — label the denominator correctly.* The mock divided by rated capacity and called it
"% of average output". With a ~30% capacity factor the two differ by a factor of three.

In [8]:
test["err_mw"] = test["gen_mw"] - test["gen_cal_mw"]
by_month = test.groupby("month")["err_mw"].agg(rmse=lambda e: np.sqrt((e ** 2).mean()), bias="mean").round(0)
by_month.T

month,1,2,3,4,5,6,7,8,9,10,11,12
rmse,1845.0,1406.0,1434.0,1607.0,1762.0,1618.0,1729.0,1746.0,1701.0,1490.0,1778.0,1500.0
bias,512.0,-202.0,-379.0,-466.0,-55.0,241.0,-180.0,-464.0,-26.0,-394.0,-417.0,-664.0


*Fix 9 — daily energy is a sum over all hours, not a mean over scheduled hours times 24.*
Dropping below-cut-in hours and then multiplying the mean by 24 overstates energy. Sum the
hourly MW (hourly data: MW·1h = MWh) and check each day has 24 rows.

In [9]:
g = m.set_index("time")["gen_mw"]
daily = g.resample("D").agg(["sum", "count"])
assert (daily["count"] == 24).all()
print(f"annual energy 2023: {daily.loc['2023', 'sum'].sum() / 1e6:.2f} TWh  (mock: 69.21 TWh)")

annual energy 2023: 25.16 TWh  (mock: 69.21 TWh)


*Fix 10 — revenue at the hourly price, not the average price.* Wind depresses the price when
it blows (corr −0.30 here), so a wind fleet earns less than the average price. This is the
capture-price / cannibalisation effect and it is part of any PPA quote.

In [10]:
price = df.set_index("time")["price_eur_mwh"]
rev_hourly = (g * price)["2023"].sum()
rev_avg = g["2023"].sum() * price["2023"].mean()
print(f"2023 revenue at hourly price: {rev_hourly / 1e6:,.0f} m EUR;  at average price: {rev_avg / 1e6:,.0f} m EUR")
print(f"capture price / average price: {rev_hourly / g['2023'].sum() / price['2023'].mean():.3f}")

2023 revenue at hourly price: 1,903 m EUR;  at average price: 2,132 m EUR
capture price / average price: 0.892


*Fix 11 — exposure is a realised cost, not RMSE × average premium.* Being short when the
system is short (low wind) costs the buy premium; being long when the system is long (high
wind) earns the sell price, which is *lower* than day-ahead. Both premia are larger exactly
when the fleet's error is large, because the fleet *is* a large part of the system.
Simulate premia that scale with system wind and settle the actual error hour by hour.

In [11]:
t = test.set_index("time")
wind_z = ((df.set_index("time")["v_hub"] - df["v_hub"].mean()) / df["v_hub"].std()).reindex(t.index)
buy_prem = rng.lognormal(np.log(20), 0.5, len(t)) * np.exp(-0.8 * wind_z)      # dearer to buy when calm
sell_disc = rng.lognormal(np.log(15), 0.5, len(t)) * np.exp(0.8 * wind_z)      # bigger haircut when windy
short = np.clip(t["gen_cal_mw"] - t["gen_mw"], 0, None).to_numpy()   # forecast more than delivered -> buy shortfall
long_ = np.clip(t["gen_mw"] - t["gen_cal_mw"], 0, None).to_numpy()   # delivered more -> sell surplus at a discount
cost = (short * buy_prem + long_ * sell_disc).sum()
# counterfactual: same errors, same premia, but independent (shuffle the premia)
cost_indep = np.mean([(short * rng.permutation(buy_prem) + long_ * rng.permutation(sell_disc)).sum() for _ in range(20)])
naive = rmse(t["gen_mw"], t["gen_cal_mw"]) * np.mean(np.r_[buy_prem, sell_disc]) * 8760
print(f"realised imbalance cost 2023:            {cost / 1e6:,.0f} m EUR")
print(f"same errors, premia shuffled (no corr):  {cost_indep / 1e6:,.0f} m EUR")
print(f"RMSE x average premium x 8760:           {naive / 1e6:,.0f} m EUR")
print(f"cost / hourly-price revenue: {cost / rev_hourly:.1%}")
print("corr(shortfall MW, buy premium) =", round(np.corrcoef(short, buy_prem)[0, 1], 3), "| corr(surplus MW, sell discount) =", round(np.corrcoef(long_, sell_disc)[0, 1], 3))

realised imbalance cost 2023:            508 m EUR
same errors, premia shuffled (no corr):  318 m EUR
RMSE x average premium x 8760:           396 m EUR
cost / hourly-price revenue: 26.7%
corr(shortfall MW, buy premium) = 0.229 | corr(surplus MW, sell discount) = 0.536


## Results (honest)

In [12]:
summary = pd.Series({
    "capacity factor (fleet curve)": round(cf, 3),
    "forecast RMSE 2023, % of capacity": res.loc["gen_cal_mw", "% of capacity"],
    "forecast RMSE 2023, % of mean output": res.loc["gen_cal_mw", "% of mean output"],
    "persistence RMSE, % of capacity": res.loc["persist_mw", "% of capacity"],
    "annual energy 2023 (TWh)": round(daily.loc["2023", "sum"].sum() / 1e6, 2),
    "capture / average price": round(rev_hourly / g["2023"].sum() / price["2023"].mean(), 3),
    "realised imbalance cost (m EUR)": round(cost / 1e6),
    "cost as % of revenue": round(cost / rev_hourly, 3),
})
print(summary.to_string())

capacity factor (fleet curve)             0.310
forecast RMSE 2023, % of capacity         0.164
forecast RMSE 2023, % of mean output      0.572
persistence RMSE, % of capacity           0.287
annual energy 2023 (TWh)                 25.160
capture / average price                   0.892
realised imbalance cost (m EUR)         508.000
cost as % of revenue                      0.267
